In [ ]:
import numpy
import pandas
import seaborn

from matplotlib import pyplot

from fitting import TIMESERIES_TABLE_PATH
from realistic import PARAMS

In [ ]:
pyplot.rcParams["figure.figsize"] = [12.8, 9.6]

In [ ]:
def err(obs, model):
    scale = numpy.abs(obs)
    scale[scale == 0] = 1
    return numpy.abs(obs - model) / scale

In [ ]:
params = {
    'a': 'fitted_a',
    'b': 'fitted_b',
    'e': 'fitted_e',
    'periapsis_phase': 'fitted_periapsis_phase',
    'Macc': 'fitted_Macc',
    'Porb': 'fitted_porb',
    'tau_eff': 'fitted_tau_eff',
}

In [ ]:
results = pandas.read_pickle(TIMESERIES_TABLE_PATH)

In [ ]:
results

In [ ]:
def plot_density(param):
    pyplot.hist2d(results[param], results[params[param]], bins=50, cmap='viridis',
                norm=pyplot.matplotlib.colors.LogNorm())
    pyplot.xlabel(param)
    pyplot.ylabel(params[param])
    pyplot.colorbar(label='Density')
    pyplot.show()

In [ ]:
plot_density('a')

In [ ]:
plot_density('b')

In [ ]:
plot_density('Porb')

In [ ]:
plot_density('Macc')

In [ ]:
plot_density('tau_eff')

In [ ]:
def plot_relation(x, y='RV accuracy', logx=True, logy=True, bins=20):
    for cadence, cadence_group in results.groupby('cadence'):
        baselines = numpy.sort(cadence_group['baseline'].unique())
        fig, axes = pyplot.subplots(1, len(baselines), figsize=(6 * len(baselines), 5), sharey=True)
        if len(baselines) == 1:
            axes = [axes]
        for ax, baseline in zip(axes, baselines):
            subset = cadence_group[cadence_group['baseline'] == baseline]
            xdata = subset[x]
            ydata = subset[y]
            diff = numpy.abs(subset[x] - subset[params[x]]) / subset[x]
            # Bin edges
            if logx:
                xbins = numpy.logspace(numpy.log10(xdata.min()), numpy.log10(xdata.max()), bins + 1)
            else:
                xbins = numpy.linspace(xdata.min(), xdata.max(), bins + 1)
            if logy:
                ybins = numpy.logspace(numpy.log10(ydata.min()), numpy.log10(ydata.max()), bins + 1)
            else:
                ybins = numpy.linspace(ydata.min(), ydata.max(), bins + 1)
            # Digitize to get bin indices
            xinds = numpy.digitize(xdata, xbins) - 1
            yinds = numpy.digitize(ydata, ybins) - 1
            mean_diff = numpy.full((bins, bins), numpy.nan)
            for i in range(bins):
                for j in range(bins):
                    mask = (xinds == i) & (yinds == j)
                    if numpy.any(mask):
                        mean_diff[i, j] = diff[mask].mean()
            color_norm = pyplot.matplotlib.colors.LogNorm(
                vmin=numpy.nanmin(mean_diff), vmax=numpy.nanmax(mean_diff)
            )
            mesh = ax.pcolormesh(
                xbins, ybins, mean_diff.T,
                cmap='viridis',
                norm=color_norm
            )
            ax.set_title(f'Cadence: {cadence}, Baseline: {baseline}')
            ax.set_xlabel(x)
            ax.set_ylabel(y)
            if logx:
                ax.set_xscale('log')
            if logy:
                ax.set_yscale('log')
            fig.colorbar(mesh, ax=ax, label='Mean absolute difference (%)')
        pyplot.tight_layout()
        pyplot.show()

In [ ]:
plot_relation('a')

In [ ]:
plot_relation('Porb')

In [ ]:
plot_relation('Macc', logx=False)

In [ ]:
plot_relation('b', logx=False)

In [ ]:
def plot_density_by_param_value(param, filter_param, bins=50, max_subplots=3, cmap='viridis'):
    values = numpy.sort(results[filter_param].dropna().unique())
    if len(values) > max_subplots:
        groups = pandas.cut(results[filter_param], bins=max_subplots, include_lowest=True)
        plot_values = [
            value for value in groups.cat.categories if numpy.any(groups == value)
        ]
        get_subset = lambda value: results[groups == value]
        format_title = lambda value: (
            f'{filter_param}: {value.left:g} to {value.right:g}'
        )
    else:
        plot_values = values
        get_subset = lambda value: results[results[filter_param] == value]
        format_title = lambda value: f'{filter_param}: {value}'

    fig, axes = pyplot.subplots(
        1, len(plot_values), figsize=(6 * len(plot_values), 5), sharex=True, sharey=True
    )
    if len(plot_values) == 1:
        axes = [axes]
    for ax, value in zip(axes, plot_values):
        subset = get_subset(value)
        hist = ax.hist2d(
            subset[param], subset[params[param]], bins=bins, cmap=cmap,
            norm=pyplot.matplotlib.colors.LogNorm()
        )
        ax.set_title(format_title(value))
        ax.set_xlabel(param)
        ax.set_ylabel(params[param])
        fig.colorbar(hist[3], ax=ax, label='Density')
    pyplot.tight_layout()
    pyplot.show()


In [ ]:
plot_density_by_param_value('Macc', 'cadence')
plot_density_by_param_value('Macc', 'baseline')
plot_density_by_param_value('Macc', 'RV accuracy')


## Parameter recovery requirements

These plots summarise which observing setups recover a chosen fitted parameter accurately. The heatmap grid shows the fraction of accurate fits in bins of true parameter value and RV accuracy for each cadence/baseline pair. The envelope plot compresses that into the worst RV accuracy that still reaches the target success fraction for each parameter bin.


In [ ]:
def _param_recovery_table(param, error_threshold=0.1):
    recovery = results.copy()
    fitted_param = params[param]
    recovery[param] = pandas.to_numeric(recovery[param], errors='coerce')
    recovery[fitted_param] = pandas.to_numeric(recovery[fitted_param], errors='coerce')
    recovery[f'{param} error'] = err(recovery[param], recovery[fitted_param])
    recovery[f'accurate_{param}'] = recovery[f'{param} error'] <= error_threshold
    return recovery.dropna(subset=[param, fitted_param, 'RV accuracy'])


def _log_edges(data, bins):
    data = pandas.to_numeric(data, errors='coerce')
    data = data[data > 0]
    return numpy.logspace(numpy.log10(data.min()), numpy.log10(data.max()), bins + 1)


def _binned_success_fraction(subset, param, xedges, yedges):
    accurate_counts, _, _ = numpy.histogram2d(
        subset[param], subset['RV accuracy'], bins=[xedges, yedges],
        weights=subset[f'accurate_{param}']
    )
    total_counts, _, _ = numpy.histogram2d(
        subset[param], subset['RV accuracy'], bins=[xedges, yedges]
    )
    success_fraction = numpy.full_like(accurate_counts, numpy.nan, dtype=float)
    numpy.divide(
        accurate_counts, total_counts, out=success_fraction, where=total_counts > 0
    )
    return success_fraction, total_counts


In [ ]:
def plot_param_recovery_heatmap_grid(param, error_threshold=0.1, param_bins=12, rv_bins=12):
    recovery = _param_recovery_table(param, error_threshold=error_threshold)
    cadences = numpy.sort(recovery['cadence'].unique())
    baselines = numpy.sort(recovery['baseline'].unique())
    xedges = _log_edges(recovery[param], param_bins)
    yedges = _log_edges(recovery['RV accuracy'], rv_bins)

    fig, axes = pyplot.subplots(
        len(baselines), len(cadences), figsize=(5 * len(cadences), 4 * len(baselines)),
        sharex=True, sharey=True
    )
    axes = numpy.atleast_2d(axes)
    for row, baseline in enumerate(baselines):
        for col, cadence in enumerate(cadences):
            ax = axes[row, col]
            subset = recovery[
                (recovery['baseline'] == baseline) & (recovery['cadence'] == cadence)
            ]
            success_fraction, total_counts = _binned_success_fraction(
                subset, param, xedges, yedges
            )
            mesh = ax.pcolormesh(
                xedges, yedges, success_fraction.T, vmin=0, vmax=1, cmap='viridis'
            )
            ax.set_xscale('log')
            ax.set_yscale('log')
            ax.set_title(f'cadence={cadence}, baseline={baseline}')
            if row == len(baselines) - 1:
                ax.set_xlabel(param)
            if col == 0:
                ax.set_ylabel('RV accuracy')

    fig.suptitle(f'{param} recovery fraction, relative error <= {error_threshold:g}')
    fig.colorbar(mesh, ax=axes, label='Accurate-fit fraction', shrink=0.9)
    pyplot.show()


In [ ]:
def plot_param_rv_accuracy_requirement(
    param, error_threshold=0.1, success_threshold=0.8, param_bins=12, rv_bins=12
):
    recovery = _param_recovery_table(param, error_threshold=error_threshold)
    cadences = numpy.sort(recovery['cadence'].unique())
    baselines = numpy.sort(recovery['baseline'].unique())
    xedges = _log_edges(recovery[param], param_bins)
    yedges = _log_edges(recovery['RV accuracy'], rv_bins)
    xcentres = numpy.sqrt(xedges[:-1] * xedges[1:])
    ycentres = numpy.sqrt(yedges[:-1] * yedges[1:])

    fig, axes = pyplot.subplots(
        1, len(cadences), figsize=(6 * len(cadences), 5), sharex=True, sharey=True
    )
    if len(cadences) == 1:
        axes = [axes]
    for ax, cadence in zip(axes, cadences):
        for baseline in baselines:
            subset = recovery[
                (recovery['baseline'] == baseline) & (recovery['cadence'] == cadence)
            ]
            success_fraction, total_counts = _binned_success_fraction(
                subset, param, xedges, yedges
            )
            required_accuracy = numpy.full(len(xcentres), numpy.nan)
            for i in range(len(xcentres)):
                good = (success_fraction[i, :] >= success_threshold) & (total_counts[i, :] > 0)
                if numpy.any(good):
                    required_accuracy[i] = numpy.nanmax(ycentres[good])
            ax.plot(xcentres, required_accuracy, marker='o', label=f'baseline={baseline}')
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_title(f'cadence={cadence}')
        ax.set_xlabel(param)
        ax.grid(True, which='both', alpha=0.25)
    axes[0].set_ylabel('Worst RV accuracy meeting target')
    axes[-1].legend(title='Baseline')
    fig.suptitle(
        f'RV accuracy requirement for {param} error <= {error_threshold:g} '
        f'in at least {success_threshold:g} of fits'
    )
    pyplot.show()


In [ ]:
for param in ['Macc', 'a', 'Porb', 'b']:
    plot_param_recovery_heatmap_grid(param, error_threshold=0.1)
    plot_param_rv_accuracy_requirement(param, error_threshold=0.1, success_threshold=0.8)
